# QLoRA Fine-Tuning — TinyLlama 1.1B

Fine-tune **TinyLlama/TinyLlama-1.1B-Chat-v1.0** with QLoRA on a custom instruction dataset.

| Param | Value |
|---|---|
| Model | TinyLlama-1.1B-Chat-v1.0 |
| LoRA rank (r) | 16 |
| LoRA alpha | 32 |
| Quantization | 4-bit NF4 |
| Learning rate | 2e-4 |
| Batch size | 4 |
| Epochs | 3 |

> **Before running:** Runtime → Change runtime type → **T4 GPU**


In [ ]:
# Cell 1 — Install dependencies
!pip install -q torch transformers datasets peft accelerate bitsandbytes sentencepiece
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
# Cell 2 — Imports
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# Cell 3 — Config
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_LEN    = 512
BATCH_SIZE = 4
EPOCHS     = 3
LR         = 2e-4

print(f"Model: {MODEL_NAME}")
print(f"Max seq length: {MAX_LEN} | Batch: {BATCH_SIZE} | Epochs: {EPOCHS} | LR: {LR}")


In [ ]:
# Cell 4 — 4-bit QLoRA config + load tokenizer & model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

print(f"Model loaded: {MODEL_NAME}")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")


In [ ]:
# Cell 5 — LoRA setup (r=16, alpha=32)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# Cell 6 — Upload train.jsonl and load dataset
from google.colab import files

print("Upload train.jsonl:")
files.upload()   # file lands at /content/train.jsonl

dataset = load_dataset(
    "json",
    data_files={"train": "/content/train.jsonl"},
)

print(f"\nLoaded: {len(dataset['train'])} samples")
print("Sample:", dataset["train"][0])


In [ ]:
# Cell 7 — Format and tokenize dataset
def format_example(example):
    inp_block = f"\n### Input:\n{example['input']}" if example.get("input", "").strip() else ""
    prompt = (
        f"### Instruction:\n{example['instruction']}"
        f"{inp_block}\n\n"
        f"### Response:\n{example['output']}"
    )
    tokens = tokenizer(
        prompt,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(
    format_example,
    remove_columns=dataset["train"].column_names,
)

print(f"Tokenized: {len(tokenized_dataset['train'])} samples")
print("Keys:", list(tokenized_dataset["train"][0].keys()))


In [ ]:
# Cell 8 — Training arguments
training_args = TrainingArguments(
    output_dir="/content/qlora-output",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    logging_steps=50,
    save_steps=200,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
)

print(training_args)


In [ ]:
# Cell 9 — Build Trainer and train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

train_result = trainer.train()
print(train_result.metrics)


In [ ]:
# Cell 10 — Save adapter weights
import os

ADAPTER_DIR = "/content/adapters"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Training complete. Adapter saved to", ADAPTER_DIR)
for fname in sorted(os.listdir(ADAPTER_DIR)):
    sz = os.path.getsize(os.path.join(ADAPTER_DIR, fname))
    print(f"  {fname}  ({sz / 1024:.1f} KB)")
